# Create Gulf of Mexico Research Initiative Awards

273 funded projects swept from the research.gulfresearchinitiative.org pid pages (Deepwater-Horizon program, 2010-20). title+abstract+PI+institution 100%, funding source as scheme. GoMRI internal admin/legacy/synthesis projects filtered out. Amounts/years not published -> NULL (§6.7 waiver).

provenance `gomri`, priority 331. F4320309630 (Path A).

Source parquet: s3://openalex-ingest/awards/gomri/gomri_projects.parquet
Built by scripts/local/gomri_to_s3.py

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gomri_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gomri/gomri_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.gomri_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.gomri_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gomri_raw LIMIT 5;

## Step 2: Create GoMRI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gomri_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309630  -- Gulf of Mexico Research Initiative
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.funder_scheme as funder_scheme,
        'gomri' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        CAST(NULL AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.gomri_raw g
    CROSS JOIN the_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gomri' AND priority = 331;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    331 as priority
FROM openalex.awards.gomri_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.gomri_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year FROM openalex.awards.gomri_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) cnt FROM openalex.awards.gomri_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(display_name) has_title, COUNT(amount) has_amount, COUNT(lead_investigator) has_pi,
  ROUND(try_divide(COUNT(amount),COUNT(*))*100,1) pct_amount FROM openalex.awards.gomri_awards;

In [ ]:
%sql
-- Step 6.4a PI frequency
SELECT lead_investigator.given_name given, lead_investigator.family_name family, COUNT(*) n
FROM openalex.awards.gomri_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name institution, COUNT(*) cnt FROM openalex.awards.gomri_awards
WHERE lead_investigator.affiliation.name IS NOT NULL GROUP BY 1 ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
SELECT COUNT(*) as gomri_in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='gomri' AND priority=331;